# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

In [ ]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

In [ ]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

In [ ]:
#lookup[(lookup['total_stocks']>0)].to_excel('manual_data.xlsx')

In [ ]:
#lookup.to_excel('manual_data.xlsx')

In [ ]:
#lookup[(lookup['brand']=='حلو الشام')&(lookup['total_stocks']>0)]

In [ ]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(6396,701,'market_min'),
(83,1126,'market_25'),
(21703,701,'target_margin'),
(2424,701,'market_min'),
(152,701,'market_min'),
(12496,700,'market_min'),
(151,704,'market_min'),
(98,1124,'market_min'),
(990,701,'market_25'),
(11427,701,'market_min'),
(11411,704,'market_50'),
(307,704,'market_min'),
(161,700,'market_min'),
(23040,701,'market_min'),
(149,703,'market_min'),
(10597,703,'market_min'),
(1548,700,'market_min'),
(305,704,'market_min'),
(8600,701,'market_min'),
(411,700,'market_25'),
(2109,701,'market_min'),
(6341,700,'market_min'),
(151,701,'market_25'),
(6342,700,'market_min'),
(147,701,'market_25'),
(2188,701,'market_min'),
(3840,1123,'market_min'),
(22968,703,'market_min'),
(82,700,'market_min'),
(82,701,'market_min'),
(19687,1123,'market_min'),
(3893,702,'market_min'),
(2991,701,'market_25'),
(286,701,'market_25'),
(148,701,'market_25'),
(13259,704,'market_25'),
(3430,1126,'market_min'),
(206,1123,'market_min'),
(410,704,'market_min'),
(147,704,'market_25'),
(305,700,'market_min'),
(305,703,'market_min'),
(149,701,'market_25'),
(24257,703,'market_25'),
(147,700,'market_25'),
(9239,701,'market_25'),
(9239,704,'market_25'),
(8938,701,'market_25'),
(146,700,'market_min'),
(84,1124,'market_25'),
(2422,1124,'market_25'),
(143,700,'market_25'),
(11092,700,'market_min'),
(439,700,'market_min'),
(13323,700,'market_min'),
(146,704,'market_min'),
(148,704,'market_25'),
(12721,701,'market_25'),
(6805,701,'market_min'),
(589,701,'market_50'),
(506,701,'market_25'),
(6235,701,'market_min'),
(149,700,'market_25'),
(6540,1125,'market_25'),
(145,1126,'market_min'),
(6569,701,'market_min'),
(21712,704,'market_min'),
(1766,700,'market_min'),
(3059,703,'market_min'),
(434,701,'market_50'),
(6220,701,'market_min'),
(248,1124,'market_50'),
(435,701,'market_75'),
(25958,701,'market_25'),
(8216,700,'market_min'),
(145,1123,'market_25'),
(2335,701,'market_50'),
(148,700,'market_25'),
(5850,701,'market_min'),
(2424,704,'market_50'),
(9372,703,'market_min'),
(344,700,'market_min'),
(12494,700,'market_min'),
(23380,704,'market_min'),
(202,704,'market_25'),
(143,704,'market_25'),
(12979,1126,'market_min'),
(4678,700,'market_min'),
(2423,704,'market_50'),
(9544,701,'market_50'),
(10529,701,'market_25'),
(3670,1125,'market_min'),
(5647,701,'market_min'),
(12637,702,'market_min'),
(6221,701,'market_min'),
(6562,700,'market_min'),
(146,703,'market_min'),
(202,701,'market_50'),
(9288,1123,'market_min'),
(2497,701,'market_25'),
(1504,701,'market_25'),
(62,701,'market_min'),
(10720,1124,'market_min'),
(11785,701,'market_25'),
(8089,700,'market_min'),
(23040,1124,'market_25'),
(24260,704,'market_25'),
(6236,701,'market_min'),
(13037,1124,'market_min'),
(25994,701,'market_25'),
(144,701,'market_50'),
(4076,704,'market_min'),
(12227,701,'market_75'),
(10595,703,'market_25'),
(5812,700,'market_50'),
(24260,703,'market_50'),
(6936,704,'market_min'),
(11092,701,'market_25'),
(24705,1126,'market_50'),
(146,701,'market_50'),
(24257,704,'market_50'),
(1504,1124,'market_25'),
(6259,700,'target_margin'),
(11683,701,'market_25'),
(11683,703,'market_25'),
(9785,701,'market_25'),
(9126,1126,'market_min'),
(24257,1123,'market_50'),
(950,701,'market_25'),
(11836,1123,'market_min'),
(11785,700,'market_25'),
(22957,700,'market_50'),
(25717,701,'target_margin'),
(25717,703,'target_margin'),
(9827,700,'market_25'),
(9828,700,'market_25'),
(144,700,'market_50'),
(6968,701,'market_25'),
(8853,701,'market_min'),
(25438,700,'market_50'),
(145,701,'market_50'),
(23245,701,'market_min'),
(6560,701,'market_min'),
(19687,701,'market_25'),
(11093,701,'market_50'),
(21476,700,'market_min'),
(10530,701,'market_50'),
(6225,701,'market_min'),
(109,701,'market_50'),
(3722,700,'market_75'),
(6799,700,'market_min'),
(8274,1125,'market_50'),
(8234,1123,'market_min'),
(3983,703,'market_25'),
(10638,700,'market_25'),
(3670,1123,'market_25'),
(11091,701,'market_50'),
(9289,701,'market_min'),
(11838,1125,'market_min'),
(11677,703,'market_25'),
(6236,700,'market_25'),
(506,700,'market_50'),
(346,702,'market_min'),
(25960,701,'market_50'),
(13481,704,'market_min'),
(2099,701,'market_min'),
(24258,703,'market_50'),
(9372,701,'market_25'),
(410,701,'market_50'),
(9784,700,'market_50'),
(21133,701,'market_min'),
(24257,701,'market_75'),
(8675,701,'market_75'),
(2835,700,'market_min'),
(5642,701,'market_25'),
(10530,700,'market_75'),
(23877,701,'market_25'),
(24260,700,'market_75'),
(8649,703,'market_25'),
(2733,701,'market_50'),
(12870,701,'market_50'),
(11306,700,'market_50'),
(8944,701,'market_75'),
(4032,1123,'market_25'),
(76,1125,'market_min'),
(22883,700,'market_50'),
(5976,1123,'market_min'),
(5208,700,'market_50'),
(13211,700,'market_50'),
(11091,700,'market_50'),
(7007,700,'market_50'),
(11093,700,'market_50'),
(11081,1125,'market_25'),
(7007,701,'market_50'),
(346,1123,'market_min'),
(10669,701,'market_50'),
(7702,700,'market_75'),
(10181,700,'market_25'),
(7513,701,'market_min'),
(12647,701,'market_50'),
(12764,701,'market_75'),
(11586,701,'market_25'),
(12993,704,'market_min'),
(13041,701,'market_50'),
(2744,700,'market_25'),
(3840,702,'market_50'),
(6197,700,'market_25'),
(6561,701,'market_min'),
(9288,701,'market_25'),
(4076,701,'market_75'),
(9186,703,'market_75'),
(27214,701,'market_25'),
(6225,700,'market_50'),
(24258,701,'market_75'),
(13246,700,'market_75'),
(3390,701,'market_50'),
(22363,700,'market_25'),
(74,701,'market_25'),
(8139,701,'market_min'),
(12535,702,'market_75'),
(23278,701,'market_50'),
(6560,700,'market_50'),
(10484,701,'market_50'),
(9126,1124,'market_25'),
(9289,700,'market_25'),
(346,700,'market_25'),
(7513,700,'market_min'),
(4032,700,'market_50'),
(8904,701,'target_margin'),
(11936,701,'market_75'),
(1668,701,'market_50'),
(11942,1123,'market_50'),
(10597,701,'market_50'),
(6864,701,'market_50'),
(346,701,'market_25'),
(7522,701,'market_25'),
(7521,701,'market_50'),
(8904,700,'target_margin'),
(24261,700,'market_75'),
(13484,700,'market_50'),
(13266,701,'market_75'),
(21703,1125,'target_margin'),
(8139,700,'market_25'),
(8510,700,'target_margin'),
(74,700,'market_75'),
(1604,700,'market_75'),
(11180,1125,'market_75'),
(9126,700,'market_75')

]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

In [ ]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
#df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

In [ ]:
# mona_data = pd.read_excel('mona_data.xlsx')
# for cohort_id in mona_data.NEW_COHORT_ID.unique():
#     out = mona_data[mona_data['NEW_COHORT_ID'] == cohort_id]
#     id_ = cohort_id
#     name = out.NEW_COHORT_NAME.values[0]
#     file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
#     out.to_excel(file_name_,index = False,engine = 'xlsxwriter')
#     time.sleep(3)
#     ################### Loop to avoid file limit ######################
#     # split file into chunks
#     print('Spliting file into chunks...')
#     if id_ == 61:
#         chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
#     else:
#         chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
#     print(f'len chunks = {len(chunks)}')
#     fileslist = []
#     for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
#         fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
#         output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
#         chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
#     # Loop over chunks and upload
#     print('Uploading...')
#     for file in fileslist:
#         chunk = file.split('chunk_')[1].split('.xls')[0]
#         x = post_prices(id_, file)
#         # print(str(x.content))
#         if ('"success":true' in str(x.content).lower()):
#             print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
#         else:
#             print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
#             print(x.content)
#             final_status = False
#             break

In [ ]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')